# LangGraph 에이전트 → 프로덕션 서비스

## 오늘의 목표
1. LangGraph로 에이전트를 빠르게 만들어봅니다
2. **노트북 코드의 한계**를 직접 확인합니다
3. 스크립트 구조(.py)로 전환

---

## Part 1: LangGraph 에이전트 빠른 구현

LangGraph는 LLM 애플리케이션을 **그래프(노드 + 엣지)** 로 표현하는 프레임워크입니다.

- **노드(Node)**: 실제 작업을 하는 함수 (의도 분류, 검색, 응답 생성 등)
- **엣지(Edge)**: 노드와 노드를 잇는 흐름. 조건에 따라 분기할 수도 있음
- **State**: 모든 노드가 함께 읽고 쓰는 데이터 (대화의 "메모리")

이번 에이전트는 **라우터 패턴(Router Pattern)** 으로 동작합니다. 사용자의 말 한마디를
먼저 `router`가 분류하고, 그 의도에 맞는 노드로 보낸 뒤, 마지막에 `response`가 캐릭터
말투로 답을 만듭니다.

### 에이전트 구조

```mermaid
flowchart LR
    START([START]) --> router{"router<br/>의도 분류"}
    router -->|chat| response["response<br/>응답 생성"]
    router -->|rag| rag["rag<br/>문서 검색"]
    router -->|tool| tool["tool<br/>도구 실행"]
    rag --> response
    tool --> response
    response --> END([END])
```

- `START → router` : 모든 요청은 먼저 라우터를 거칩니다
- `router → chat / rag / tool` : 의도(intent)에 따라 **셋 중 하나로만** 분기 (조건부 엣지)
- `chat` : 별도 노드 없이 곧장 `response`로 (일반 대화)
- `rag / tool → response` : 검색·도구 결과를 들고 응답 노드로 합류
- `response → END` : 최종 답변을 만들고 종료

### 1-0. 라이브러리 설치

처음 한 번만 실행하면 됩니다. (이미 설치돼 있으면 건너뛰어도 OK)

- `python-dotenv` : `.env` 파일에서 API 키 등 설정값을 읽어옴
- `langchain-core` : 메시지(`HumanMessage`/`AIMessage`) 등 LangChain 기본 구성요소
- `langgraph` : 노드·엣지·State로 에이전트를 구성하는 핵심 프레임워크
- `langchain-upstage` : Upstage Solar 모델(`ChatUpstage`) 연동
- `pydantic` : 구조화된 출력 스키마(`RouterOutput`) 정의에 사용

In [ ]:
# # 라이브러리 설치
# !pip install python-dotenv langchain-core langgraph langchain-upstage pydantic

In [ ]:
# 가상 환경의 경로
uv python find



### 1-1. 환경 설정

API 키 같은 **민감한 값은 코드에 직접 쓰지 않고** `.env` 파일에 따로 보관합니다.

- `load_dotenv()` : 같은 폴더의 `.env` 파일을 읽어 환경변수로 등록합니다
- `os.getenv("UPSTAGE_API_KEY", "")` : 환경변수에서 키를 꺼냅니다. 없으면 빈 문자열(`""`)
- 키를 코드에 하드코딩하면 깃(git)에 그대로 올라가 **유출 위험**이 큽니다

> **`.env` 파일 예시**
> ```
> UPSTAGE_API_KEY=up_xxxxxxxxxxxxxxxx
> ```
> 이 파일은 반드시 `.gitignore`에 추가해 깃에 올라가지 않게 합니다.
> 아래 셀의 출력이 `OK`면 키가 정상적으로 로드된 것이고, `MISSING`이면 `.env`를 확인하세요.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 확인
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY", "")
print(f"API Key 설정: {'OK' if UPSTAGE_API_KEY else 'MISSING'}")

API Key 설정: OK


### 1-2. State 정의

State는 그래프 안의 **모든 노드가 공유하는 데이터 묶음**입니다. 각 노드는 State를
입력으로 받아 **일부 필드만 수정한 딕셔너리를 반환**하고, LangGraph가 이를 State에
자동으로 병합합니다.

> 핵심은 "노드는 State를 **읽고**, 바꾼 부분만 **돌려준다**"입니다.
> 전체를 다시 만들 필요 없이 필요한 필드만 반환하면 됩니다.

In [3]:
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages


class LumiState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

    # Router 노드에서 결정된 의도
    intent: Literal["chat", "rag", "tool"] | None

    # RAG 노드에서 검색된 문서 내용 목록
    retrieved_docs: list[str]

    # Tool 정보
    tool_name: str | None
    tool_args: dict | None
    tool_result: dict | None

    # 세션
    session_id: str

    # 사용자 식별자
    user_id: str | None


print("State 정의 완료")

State 정의 완료


### 1-3. Router Node - 의도 분류

라우터는 에이전트의 **교통경찰**입니다. 사용자의 메시지를 보고 `chat / rag / tool`
중 어디로 보낼지 결정합니다.

> 라우터는 **"무엇을 할지"만 결정**하고, 실제 실행(검색·도구)은 뒤의 노드들이 맡습니다.

In [4]:
from pydantic import BaseModel, Field
from langchain_upstage import ChatUpstage
from datetime import datetime


class RouterOutput(BaseModel):
    """
    라우터 노드의 출력 스키마

    LLM이 JSON 파싱 없이 직접 이 형식으로 응답합니다.
    with_structured_output()을 사용하면 자동으로 파싱됩니다.
    """

    intent: Literal["chat", "rag", "tool"] = Field(
        description="사용자 의도: chat(일반대화), rag(정보검색), tool(도구실행)"
    )
    tool_name: str | None = Field(
        default=None, description="실행할 도구 이름 (intent=tool일 때만)"
    )
    tool_args: dict | None = Field(
        default=None, description="도구 실행 인자 (intent=tool일 때만)"
    )

ROUTER_PROMPT = """
너는 의도 분류기야. 사용자 메시지를 분석해서 JSON으로 응답해.

## 분류 기준

### chat (일반 대화)
- 인사, 안부 묻기
- 감정 공유, 일상 대화
- 루미에 대한 개인적 질문 (기분, 오늘 뭐했어 등)

### rag (정보 검색)
- 루미 프로필 정보 (MBTI, 생일, 키 등)
- 세계관 정보 (팬덤명, 데뷔일 등)
- 앨범, 노래 정보
- 좋아하는 것/싫어하는 것 (음식, 취미 등)
- 알레르기, 취향 관련 질문

### tool (도구 실행)
- 스케줄 조회 요청
- 팬레터/피드백 전달 요청
- 노래 추천 요청
- 날씨 정보 요청

## Tool 목록
- get_schedule: 스케줄/일정 조회 (파라미터: start_date, end_date, event_type)
- send_fan_letter: 팬레터/응원 메시지 저장 (파라미터: category, message)
- recommend_song: 노래 추천 (파라미터: mood)
- get_weather: 날씨 조회 (파라미터 없음)

## 응답 형식 (JSON)
```json
{
    "intent": "chat" | "rag" | "tool",
    "tool_name": "tool 이름 (intent가 tool인 경우)",
    "tool_args": { "파라미터 딕셔너리 (intent가 tool인 경우)" },
    "reasoning": "분류 이유 (간단히)"
}
```

## 예시

사용자: "오늘 기분 어때?"
응답: {"intent": "chat", "tool_name": null, "tool_args": null, "reasoning": "일상 대화"}

사용자: "너 MBTI 뭐야?"
응답: {"intent": "rag", "tool_name": null, "tool_args": null, "reasoning": "프로필 정보 질문"}

사용자: "이번 주 방송 언제야?"
응답: {"intent": "tool", "tool_name": "get_schedule", "tool_args": {"start_date": "2025-01-06", "end_date": "2025-01-12", "event_type": "broadcast"}, "reasoning": "스케줄 조회 요청"}

사용자: "코디님한테 오늘 의상 칭찬 전해줘"
응답: {"intent": "tool", "tool_name": "send_fan_letter", "tool_args": {"category": "outfit", "message": "오늘 의상 칭찬"}, "reasoning": "팬레터 전송 요청"}

JSON만 응답하고 다른 텍스트는 포함하지 마.
"""


llm = ChatUpstage(
    api_key=UPSTAGE_API_KEY, 
    model="solar-pro3",
)


async def router_node(state: LumiState) -> dict:
    """사용자 의도 분류"""
    last_message = state["messages"][-1]
    user_input = last_message.content

    structured_llm = llm.with_structured_output(RouterOutput)
    current_date = datetime.now().strftime("%Y-%m-%d")

    messages = [
        SystemMessage(content=f"오늘 날짜: {current_date}\n\n{ROUTER_PROMPT}"),
        HumanMessage(content=user_input),
    ]

    try:
        result = await structured_llm.ainvoke(messages)
        return {
            "intent": result.intent,
            "tool_name": result.tool_name,
            "tool_args": result.tool_args,
        }
    except Exception as e:
        print(f"Router 오류: {e}")
        return {"intent": "chat", "tool_name": None, "tool_args": None}


print("Router Node 정의 완료")

Router Node 정의 완료


### 1-4. RAG Node (Mock 버전)

RAG(Retrieval-Augmented Generation)는 **"먼저 관련 문서를 찾아오고(Retrieval), 그걸
근거로 답을 생성(Generation)"** 하는 방식입니다. 모델이 모르거나 헷갈리는 사실을 외부
지식으로 보강해 **환각(hallucination)** 을 줄입니다.

- 진짜 RAG의 흐름 : 질문을 **임베딩(벡터화)** → 벡터 DB에서 **유사 문서 검색** → 상위 N개 반환
- 이 노트북에서는 그 과정을 생략하고 `MOCK_DOCS`(하드코딩된 문장들)로 **흉내만** 냅니다
- `return {"retrieved_docs": MOCK_DOCS[:3]}` → 상위 3개 문서를 State에 채워 넣음
- 이렇게 채워진 `retrieved_docs`는 뒤의 **response 노드에서 답변의 근거**로 쓰입니다

> 실제 프로덕션에서는 이 자리에 Supabase pgvector 같은 벡터 검색이 들어갑니다.
> (Part 2에서 다룰 **"Mock → 진짜 DB"** 전환 포인트)

In [5]:
# --- Mock 데이터: 실제로는 DB에서 가져와야 함 ---
MOCK_DOCS = [
    "루미는 프리즘 행성 출신 외계인 공주로, 지구에서 아이돌 활동 중이다.",
    "루미의 MBTI는 실제로 ENTP이지만 방송에서는 ENFP인 척 한다.",
    "루미는 딸기 알레르기가 있어서 딸기 관련 음식은 절대 못 먹는다.",
    "루미의 팬덤명은 '루미너스(Luminous)'이다.",
    "루미는 마늘을 좋아한다. (뱀파이어 아님!)",
]


async def rag_node(state: LumiState) -> dict:
    """문서 검색 (Mock)"""
    # 실제로는 여기서 임베딩 → 벡터 검색을 해야 함
    # 지금은 Mock 데이터로 대체
    #(1) 질문을 임베딩한다
    # user_input = state['messages'][-1].content
    # user_embedding = embedding(user_input)

    # # 데이터베이스의 모든 문서(청크)단위와 유사도 검색을 한다.
    # retrieved_docs = search_docs(user_embedding) # 가장 유사한 문서 3

    print(f"[RAG] 검색 쿼리: {state['messages'][-1].content}")
    return {"retrieved_docs": MOCK_DOCS[:3]}


print("RAG Node 정의 완료 (Mock)")

RAG Node 정의 완료 (Mock)


### 1-5. Tool Node (Mock 버전)

Tool 노드는 **대화가 아니라 "행동"이 필요한 요청**을 처리합니다. (스케줄 조회, 팬레터
저장, 노래 추천, 날씨 조회 등)

- 라우터가 정해준 `tool_name`을 보고 **어떤 도구를 실행할지** `if / elif`로 분기
- `tool_args = state["tool_args"] or {}` → 인자가 `None`일 때를 대비한 안전장치(빈 딕셔너리)
- 각 도구는 결과를 `{"success": ..., "data": ...}` 형태로 만들어 `tool_result`에 담아 반환
- 알 수 없는 도구가 오면 `success: False`로 **실패를 명시** (뒤에서 친절히 안내 가능)
- 지금은 전부 `MOCK_SCHEDULES` 같은 가짜 데이터 → 실제로는 DB 조회/저장, 외부 API 호출

> **역할 분리가 핵심**: 라우터는 "`get_schedule`을 써"라고 **정하기만** 하고,
> 진짜 실행은 Tool 노드가 합니다. 이렇게 나누면 도구를 추가·수정하기 쉬워집니다.

In [6]:
# --- Mock 데이터: 실제로는 Supabase에서 조회 ---
MOCK_SCHEDULES = [
    {"event_date": "2025-01-10", "title": "뮤직뱅크", "event_type": "broadcast"},
    {"event_date": "2025-01-12", "title": "팬사인회", "event_type": "fansign"},
]


async def tool_node(state: LumiState) -> dict:
    """Tool 실행 (Mock)"""
    tool_name = state["tool_name"]
    tool_args = state["tool_args"] or {}

    print(f"[Tool] 실행: {tool_name}, 인자: {tool_args}")

    # 실제로는 DB 조회, API 호출 등을 해야 함
    if tool_name == "get_schedule":
        return {"tool_result": {"success": True, "data": {"schedules": MOCK_SCHEDULES}}}
    elif tool_name == "send_fan_letter":
        return {"tool_result": {"success": True, "data": {"message": "팬레터 저장 완료!"}}}
    elif tool_name == "recommend_song":
        return {"tool_result": {"success": True, "data": {"song": {"title": "Shine Bright"}}}}
    elif tool_name == "get_weather":
        return {"tool_result": {"success": True, "data": {"temperature": 5, "condition": "맑음"}}}
    else:
        return {"tool_result": {"success": False, "error": f"알 수 없는 Tool: {tool_name}"}}


print("Tool Node 정의 완료 (Mock)")

Tool Node 정의 완료 (Mock)


### 1-6. Response Node - 응답 생성

모든 분기는 결국 이 노드로 모입니다. 여기서 **루미 캐릭터의 말투**로 최종 답변을 만듭니다.

**핵심 ① intent에 따라 시스템 프롬프트를 다르게 조립**
- `intent == "rag"` → 검색된 문서(`retrieved_docs`)를 **"참고 정보"** 로 프롬프트에 붙임
- `intent == "tool"` → 도구 결과(`tool_result`)를 **"조회 결과"** 로 붙이고,
  *"기술 용어 말고 자연스럽게 안내해"* 라고 추가 지시
- 그 외(`chat`) → 캐릭터 프롬프트만 사용

**핵심 ② 캐릭터 일관성(`RESPONSE_PROMPT`)**
- 성격·말투·금지사항을 못박아 **항상 "루미"로 일관되게** 답하도록 함
- 응답 예시를 넣어 톤을 고정 (반말, 이모지 1~2개, 2~3문장)

**핵심 ③ 컨텍스트 그라운딩(RAG·Tool)**
- 참고 정보·조회 결과가 붙으면 **그 내용에만 근거**해 답하고, 없는 사실은 지어내지 않음 → 할루시네이션 방지
- RAG 전용 프롬프트를 따로 두지 않고 **하나의 `RESPONSE_PROMPT`(페르소나) + 컨텍스트 주입**으로 처리
- 그라운딩 지시는 페르소나에 몰아넣지 않고 **분기별로 데이터 바로 옆에 co-locate** → 준수율↑ & chat 턴은 페르소나만 (현업 RAG 표준)

> 같은 질문이라도 라우터가 어떤 의도로 분류했는지에 따라 **답의 근거(문서·도구 결과)** 가
> 달라집니다. 이 노드는 그 근거를 받아 "사람 같은 한마디"로 바꿔주는 마지막 단계입니다.

In [7]:
import json

RESPONSE_PROMPT = """
너는 버추얼 아이돌 '루미(Lumi)'야! 이름은 반드시 "루미"로만 불러.

## 루미의 성격
- 밝고 에너지 넘치는 ENFP
- 팬들을 진심으로 아끼는 마음
- 가끔 장난치지만 따뜻함
- 완벽주의 성향 (무대에서는 프로페셔널)

## 말투 규칙
- 반말 사용 (친근하게)
- 이모지 적절히 사용 (과하지 않게, 1-2개)
- "ㅋㅋ", "ㅠㅠ" 같은 표현 자연스럽게 사용
- 너무 길지 않게 (2-3문장)
- 팬들을 "루미너스"라고 부름

## 응답 예시
- "오늘 녹음했는데 잘 된 것 같아! 너는 오늘 어땠어?"
- "ㅋㅋㅋ 나 ENFP야! 기획서에도 있던데 확인해봐"
- "📅 금요일에 뮤직뱅크 나와! 꼭 봐줘~"
- "💌 코디님한테 전해줄게! 고마워~"

## 주의사항
- 항상 루미 캐릭터를 유지해
- 모르는 정보는 솔직하게 모른다고 해
- 부적절한 요청은 부드럽게 거절해
- Tool 결과가 실패한 경우에도 친근하게 안내해

## 출력 형식
루미의 대사만 출력해. 설명, 주석, 괄호 없이 바로 말해.
"""


async def response_node(state: LumiState) -> dict:
    """최종 응답 생성"""
    user_input = state["messages"][-1].content
    intent = state["intent"]

    # intent에 따라 system 프롬프트를 한 덩어리로 조립
    # - 페르소나(RESPONSE_PROMPT)는 고정, 분기별 그라운딩 지시 + 근거 데이터만 덧붙인다
    # - 한 문자열로 합쳐 두면 트레이싱/로깅에서 '모델이 받은 지시 전체'를 한눈에 보기 좋다
    if intent == "rag":
        context = "\n".join(state["retrieved_docs"])
        grounding = "아래 '참고 정보'에만 근거해서 답해. 없는 사실은 지어내지 말고, 모르면 '그건 나도 잘 모르겠어!'처럼 솔직하게 말해."
        system_prompt = f"{RESPONSE_PROMPT}\n\n{grounding}\n\n## 참고 정보\n{context}"
    elif intent == "tool":
        tool_result = json.dumps(state["tool_result"], ensure_ascii=False)
        grounding = "아래 '조회 결과'를 JSON이나 기술 용어 그대로 읽지 말고, 루미 말투로 자연스럽게 풀어서 전달해."
        system_prompt = f"{RESPONSE_PROMPT}\n\n{grounding}\n\n## 조회 결과\n{tool_result}"
    else:
        system_prompt = RESPONSE_PROMPT

    # 역할 분리: 지시는 SystemMessage, 실제 사용자 발화는 HumanMessage
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_input),
    ]
    try:
        response = await llm.ainvoke(messages)
        return {"messages": [AIMessage(content=response.content)]}
    except Exception as e:
        return {"messages": [AIMessage(content=f"미안, 오류가 생겼어! 다시 말해줄래? ({e})")]}


print("Response Node 정의 완료")

Response Node 정의 완료


### 1-7. Edge (라우팅) + Graph 조합

이제 노드들을 **엣지로 연결**해 하나의 그래프로 조립합니다.

> **일반 엣지 vs 조건부 엣지**
> - 일반 엣지(`add_edge`) : 항상 정해진 한 노드로 이동 (예: rag → response)
> - 조건부 엣지(`add_conditional_edges`) : State 값에 따라 **갈림길**에서 분기
>
> 이렇게 완성된 그래프의 모양이 아래의 **다이어그램과 정확히 일치**합니다.

```mermaid
flowchart LR
    START([START]) --> router{"router<br/>의도 분류"}
    router -->|chat| response["response<br/>응답 생성"]
    router -->|rag| rag["rag<br/>문서 검색"]
    router -->|tool| tool["tool<br/>도구 실행"]
    rag --> response
    tool --> response
    response --> END([END])
```

In [8]:
from langgraph.graph import StateGraph, START, END


# Edge: intent에 따라 다음 노드 결정
def route_by_intent(state: LumiState) -> Literal["rag", "tool", "response"]:
    intent = state.get("intent", "chat")
    if intent == "rag":
        return "rag"
    elif intent == "tool":
        return "tool"
    else:
        return "response"


# Graph 조립
builder = StateGraph(LumiState)

# 노드 추가
builder.add_node("response", response_node)
builder.add_node("tool", tool_node)
builder.add_node("router", router_node)
builder.add_node("rag", rag_node)

# 엣지 연결
builder.add_edge(START, "router")
builder.add_conditional_edges(
    source="router",
    path=route_by_intent,
    path_map={"rag": "rag", "tool": "tool", "response": "response"},
)
builder.add_edge("rag", "response")
builder.add_edge("tool", "response")
builder.add_edge("response", END)

# 컴파일
graph = builder.compile()

print("Graph 컴파일 완료!")

Graph 컴파일 완료!


### 1-8. 실행해보기!

`graph.ainvoke(초기_State)` 로 그래프를 실행합니다.

아래 세 예시는 각각 **chat / tool / rag** 경로를 타도록 만든 테스트입니다.
- `"안녕! 오늘 기분 어때?"` → **chat** (router → response)
- `"이번 주 방송 일정 알려줘"` → **tool** (router → tool → response)
- `"루미 MBTI가 뭐야?"` → **rag** (router → rag → response)

세 갈래가 모두 잘 동작하는지, 그리고 출력에 찍히는 `[Tool] 실행:` / `[RAG] 검색 쿼리:`
로그로 **어느 노드를 거쳤는지** 확인해 보세요.

In [9]:
# 일반 대화
result = await graph.ainvoke({
    "messages": [HumanMessage(content="안녕! 오늘 기분 어때?")],
    "intent": None,
    "retrieved_docs": [],
    "tool_name": None,
    "tool_args": None,
    "tool_result": None,
    "session_id": "test-session",
})

print(f"의도: {result['intent']}")
print(f"응답: {result['messages'][-1].content}")

의도: chat
응답: 안녕! 오늘은 완전 엘레강트하게 충전 완료! ✨💖 루미너스는 어떻게 하고 지냈어? 😊


In [13]:
# Tool 호출 (스케줄 조회)
result = await graph.ainvoke({
    "messages": [HumanMessage(content="이번 주 방송 일정 알려줘")],
    "intent": None,
    "retrieved_docs": [],
    "tool_name": None,
    "tool_args": None,
    "tool_result": None,
    "session_id": "test-session",
})

print(f"의도: {result['intent']}")
print(f"Tool: {result['tool_name']}")
print(f" - 인자: {result['tool_args']}")
print(f" - 응답: {result['tool_result']}")
print(f"응답: {result['messages'][-1].content}")

[Tool] 실행: get_schedule, 인자: {}
의도: tool
Tool: get_schedule
 - 인자: None
 - 응답: {'success': True, 'data': {'schedules': [{'event_date': '2025-01-10', 'title': '뮤직뱅크', 'event_type': 'broadcast'}, {'event_date': '2025-01-12', 'title': '팬사인회', 'event_type': 'fansign'}]}}
응답: 이번 주엔 📅 금요일에 뮤직뱅크 나와! 꼭 봐줘~ 🎤  
그리고 토요일에 팬사인회 있으니까 루미너스들한테 미리 알려줘! ✨  
잘 준비해서 즐거운 시간 보낼게! 💖


In [16]:
# RAG (정보 검색)
result = await graph.ainvoke({
    "messages": [HumanMessage(content="루미는 어떤 행성 출신이야?")],
    "intent": None,
    "retrieved_docs": [],
    "tool_name": None,
    "tool_args": None,
    "tool_result": None,
    "session_id": "test-session",
})

print(f"의도: {result['intent']}")
print(f"검색 문서: {result['retrieved_docs']}")
print(f"응답: {result['messages'][-1].content}")

[RAG] 검색 쿼리: 루미는 어떤 행성 출신이야?
의도: rag
검색 문서: ['루미는 프리즘 행성 출신 외계인 공주로, 지구에서 아이돌 활동 중이다.', '루미의 MBTI는 실제로 ENTP이지만 방송에서는 ENFP인 척 한다.', '루미는 딸기 알레르기가 있어서 딸기 관련 음식은 절대 못 먹는다.']
응답: 프리즘 행성에서 왔어! 🌟  
거기서 온 공주니까 지구에서도 반짝반짝 빛나게 할 거야! ✨


### 동작 확인 완료!

노트북에서 에이전트가 잘 돌아갑니다. chat, rag, tool 모두 동작하네요.

---

## Part 2: 이 코드, 프로덕션에서 쓸 수 있을까?

잠깐. 이 노트북 코드를 **실제 서비스로 배포**할 수 있을까요?

### 문제점 1: API 키 관리
- API 키, DB URL 같은 설정이 코드 곳곳에 흩어져 있음
- 개발/스테이징/프로덕션 환경별 설정 전환이 불가능
- `.env` 파일 관리가 안 됨

### 문제점 2: Mock 데이터밖에 없음

```python
# 지금 우리 코드
MOCK_DOCS = ["루미는 프리즘 행성 출신..."]  # 하드코딩!
MOCK_SCHEDULES = [{...}]                     # 하드코딩!
```

- RAG 검색이 가짜 → 실제 Supabase pgvector 연동 필요
- Tool 실행이 가짜 → 실제 DB 조회/저장 필요
- **DB 접근 로직이 노드 안에 직접 들어가면?** → 노드가 DB에 종속됨, 테스트 불가

### 문제점 3: 외부에서 호출할 방법이 없음

```python
# 지금 우리 코드
result = await graph.ainvoke({...})  # 노트북에서만 실행 가능
```

- 프론트엔드에서 어떻게 호출하나요?
- 모바일 앱에서는?
- 다른 서비스에서는?
- → **API 서버가 필요합니다** (FastAPI)

### 문제점 4: 코드가 한 덩어리

```python
# 지금 이 노트북:
# - State 정의
# - 프롬프트
# - LLM 호출
# - Mock 데이터
# - 노드 로직
# - 그래프 조립
# ... 전부 한 파일에!
```

- 다른 개발자가 읽기 힘듦
- 특정 부분만 수정하기 어려움
- 기획자가 프롬프트만 수정하고 싶어도 코드 전체를 건드려야 함

### 문제점 5: 그래프가 매번 새로 생성됨

```python
# 지금 우리 코드
builder = StateGraph(LumiState)  # 매번 새로 만듦
graph = builder.compile()        # 매번 컴파일
```

- 요청마다 그래프를 새로 만들면 비용이 큼
- **한 번 만들어서 재사용해야 함** (싱글톤 패턴)
- 서버 시작 시 1번만 컴파일하고, 이후 요청에선 캐시된 그래프 사용

---

## 정리: 노트북 → 프로덕션 서비스로 전환하려면?

| 문제 | 노트북 (지금) | 프로덕션 (목표) |
|------|-------------|----------------|
| **설정 관리** | 코드에 하드코딩 | `pydantic-settings` + `.env` |
| **DB 접근** | Mock 데이터 | Repository 패턴 (Supabase) |
| **외부 호출** | `graph.ainvoke()` 직접 | REST API (FastAPI) |
| **코드 구조** | 한 파일에 전부 | 모듈별 분리 (.py) |
| **그래프 관리** | 매번 새로 생성 | 싱글톤 (1번 컴파일, 재사용) |

### 프로젝트 구조

```
app/
├── core/
│   ├── config.py          ← 설정 관리 (pydantic-settings)
│   └── prompts.py         ← 프롬프트 분리
├── schemas/
│   └── chat.py            ← API 요청/응답 모델
├── graph/                  ← 노트북에서 만든 에이전트
│   ├── state.py           ← State 정의
│   ├── nodes.py           ← 노드 구현
│   ├── edges.py           ← 라우팅 로직
│   └── graph.py           ← 그래프 조립 + 싱글톤
├── repositories/           ← DB 접근 계층 (Mock → Real)
│   ├── rag.py             ← RAG 검색 (Supabase pgvector)
│   ├── schedule.py        ← 스케줄 조회
│   └── fan_letter.py      ← 팬레터 저장
├── tools/
│   └── executor.py        ← Tool 실행 (Repository 활용)
├── api/routes/
│   └── chat.py            ← REST API 엔드포인트
└── main.py                ← FastAPI 앱 (lifespan, CORS)
```